In [1]:
import pandas as pd

In [3]:
# Import the exported CSV
# df = pd.read_csv("rag_and_flag_ai/countries_subset.csv")
df = pd.read_csv("rag_and_flag_ai/country_content.csv")

In [4]:
df.head()

,Country,Content
0,Afghanistan,"Afghanistan, officially the Islamic Emirate of..."
1,Albania,"Albania, officially the Republic of Albania, i..."
2,Algeria,"Algeria, officially the People's Democratic Re..."
3,American Samoa,American Samoa is an unincorporated and unorga...
4,Andorra,"Andorra, officially the Principality of Andorr..."


In [5]:
from langchain_community.document_loaders.csv_loader import CSVLoader

In [6]:
# loader = CSVLoader("rag_and_flag_ai/countries_subset.csv")
loader = CSVLoader("rag_and_flag_ai/country_content.csv")
data = loader.load()
# print(data)

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb

In [9]:
# Initialize splitter (adjust chunk_size and overlap as needed)
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

all_chunks = []
for idx, row in df.iterrows():
    country = row["Country"]
    text = row["Content"]
    # Split text into chunks
    chunks = splitter.split_text(text)
    # Optionally, add metadata for each chunk
    for chunk in chunks:
        all_chunks.append({"country": country, "content": chunk})

# Now, all_chunks is ready for embedding

In [10]:
# print(all_chunks)
print(len(all_chunks))

50117


In [11]:
from chromadb.utils import embedding_functions

In [12]:
# Create embedding function
embed_model_name = "intfloat/e5-base-v2"
embed_model_fn = embedding_functions.SentenceTransformerEmbeddingFunction(embed_model_name)

In [13]:
# Chroma client
chroma_client = chromadb.PersistentClient(".chroma_db")

In [14]:
# Create collection with specified embedding function
wiki_collection = chroma_client.get_or_create_collection(
    name="wikipedia",
    embedding_function=embed_model_fn
)

In [16]:
# Add documents to collection
for i, chunk in enumerate(all_chunks):
    wiki_collection.add(
        ids=[f"{chunk['country']}_{i}"],           # unique ID for each chunk
        documents=[chunk["content"]],                 # the chunk text
        metadatas=[{"country": chunk["country"]}]  # metadata
    )

In [17]:
# Check number of documents in collection
print(wiki_collection.count())

50117


In [18]:
# Example usage for querying collection
query = "This country is rich in natural resources."

results = wiki_collection.query(
    query_texts=[ query ],
    n_results=5
)

for k, v in results.items():
    print(f"{k}: {v}\n")

ids: [['Rwanda_37642', 'Kyrgyzstan_24156', 'Guyana_18409', 'Estonia_13823', 'Algeria_762']]

embeddings: None

documents: [['on food imports. Despite a fertile landscape, the country possesses limited natural resources.', "Kumtor Gold Mine and other regions. The country's plentiful water resources and mountainous terrain enable it to produce and export large quantities of hydroelectric energy.", 'has a wide variety of natural habitats and very high biodiversity. The country also hosts a part of the Amazon rainforest, the largest and most biodiverse tropical rainforest in the world.', '=== Natural resources and mining ===', '=== Oil and natural resources ===']]

uris: None

included: ['metadatas', 'documents', 'distances']

data: None

metadatas: [[{'country': 'Rwanda'}, {'country': 'Kyrgyzstan'}, {'country': 'Guyana'}, {'country': 'Estonia'}, {'country': 'Algeria'}]]

distances: [[0.24131038784980774, 0.2735154926776886, 0.2851446270942688, 0.2858692705631256, 0.28747671842575073]]

